# Bahdanau Additive Attention in NumPy


In [1]:
import numpy as np

rng = np.random.default_rng(7)
d_q, d_k, d_a, d_v, T = 4, 3, 5, 6, 5
query = rng.normal(size=d_q)
keys = rng.normal(size=(T, d_k))
values = rng.normal(size=(T, d_v))
W_q = rng.normal(size=(d_a, d_q))
W_k = rng.normal(size=(d_a, d_k))
v_vec = rng.normal(size=d_a)
queries = rng.normal(size=(3, d_q))

def stable_softmax(x, axis=-1):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def bahdanau_scores(query, keys, W_q, W_k, v_vec):
    query_part = W_q @ query
    key_parts = keys @ W_k.T
    return np.tanh(key_parts + query_part) @ v_vec

def attention_weights(scores):
    return stable_softmax(scores)

def context_vector(weights, values):
    return weights @ values

def alignment_matrix(queries, keys, W_q, W_k, v_vec):
    return np.array([
        attention_weights(bahdanau_scores(q, keys, W_q, W_k, v_vec))
        for q in queries
    ])

e = bahdanau_scores(query, keys, W_q, W_k, v_vec)
w = attention_weights(e)
print(w.shape, round(float(w.sum()), 6))
c = context_vector(w, values)
print(c.shape)
Aln = alignment_matrix(queries, keys, W_q, W_k, v_vec)
print(Aln.shape, np.allclose(Aln.sum(axis=1), 1))
print(attention_weights(bahdanau_scores(query, keys, W_q, W_k, np.zeros(5))))
